In [ ]:
!pip install datasets transformers

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [ ]:
import pandas as pd

In [ ]:
twitter_dataset = load_dataset("Alienmaster/german_politicians_twitter_sentiment")

In [ ]:
print(twitter_dataset)

In [ ]:
#Save true labels from the dataset
true_labels = twitter_dataset["test"]["majority_sentiment"]

In [ ]:
#rename labels for comparison
label_mapping = {1: "positive", 2: "negative", 3: "neutral"}

In [ ]:
true_labels = [label_mapping[label] for label in true_labels]

In [ ]:
#extract texts for evaluation
prediction_texts = twitter_dataset["test"]["text"]

In [ ]:
#test with first model: tabularisai/multilingual-sentiment-analysis
#implement model to predict labels
model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def predict_sentiment(texts):
    results = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        with torch.no_grad():
            outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        sentiment_map = {0: "negative", 1: "negative", 2: "neutral", 3: "positive", 4: "positive"}
        results.append(sentiment_map[torch.argmax(probabilities, dim=-1).item()])
    return results

In [ ]:
sentiments_multilingual_model = predict_sentiment(prediction_texts)

In [ ]:
predicted_labels = sentiments_multilingual_model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
#calculate evaluation metrics by comparing with the correct labels
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy Multilingual Sentiment Analysis: {accuracy:.4f}")

In [ ]:
report = classification_report(true_labels, predicted_labels)
print("Classification report Multilingual Sentiment Analysis:\n", report)

In [ ]:
#test with second model: german-sentiment-bert
#implement model to predict labels

In [ ]:
pip install germansentiment

In [ ]:
from germansentiment import SentimentModel

In [ ]:
model = SentimentModel()

texts = prediction_texts

result = model.predict_sentiment(texts)

In [ ]:
predicted_labels = result

In [ ]:
#Calculate evaluation metrics by comparing with the correct labels
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy German-Sentiment-Bert: {accuracy:.4f}")

In [ ]:
report = classification_report(true_labels, predicted_labels)
print("Classification Report German-Sentiment-Bert:\n", report)

In [ ]:
#test with third model: XLM-RoBERTa-German-sentiment
#implement model to predict labels

In [ ]:
model_name= "ssary/XLM-RoBERTa-German-sentiment"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
sentiment_classes = ['negative', 'neutral', 'positive']

In [ ]:
def predict_sentiment_labels(texts):
    predicted_labels = []

    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)


        with torch.no_grad():
            outputs = model(**inputs)


        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)


        predicted_label = sentiment_classes[predictions.argmax()]
        predicted_labels.append(predicted_label)

    return predicted_labels

In [ ]:
predicted_labels = predict_sentiment_labels(prediction_texts)

In [ ]:
#Calculate evaluation metrics by comparing with the correct labels
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy XLM-RoBERTa-German-sentiment: {accuracy:.4f}")

In [ ]:
report = classification_report(true_labels, predicted_labels)
print("Classification report XLM-RoBERTa-German-sentiment:\n", report)